In [ ]:
#| hide
from vishalakshi import *
from fastcore.all import *

# concepts

> encoders, shelves, backends, and when to reach past the defaults

Choose an encoder, model backend, or shelf only when the default no longer fits.


In [ ]:
from tempfile import mkdtemp
from vishalakshi import Vault

v = Vault(Path(mkdtemp())/'vault.db')
v.note('federate fuses the legs by rank because they share no vector space: the vault embeds '
       'prose, kosha embeds identifiers, ripgrep embeds nothing.', tags=['retrieval'])
v.add('# Late chunking\n\n## Method\n\nEmbed the whole document, then pool per chunk, so a chunk '
      'keeps the context around it.\n\n## Results\n\nEvaluated on BEIR.', 'Late chunking', kind='note')
v.connect()
v.stats()

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'docs': 2,
 'nodes': 6,
 'chunks': 3,
 'encoder': 'model2vec',
 'entities': 21,
 'path': '/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmph5ysy_7x/vault.db',
 'by_kind': {'note': 2}}

## What retrieval actually returns

`v.context(q)` is the primitive underneath `ask`. It returns whole sections with breadcrumbs and `node_id`s you can `read` back.


In [ ]:
c = v.context('why are rankings fused instead of distances?', sections=4, related=4)
for s in c.results: print(f'{len(s.text):5}  {s.breadcrumb}')

  141  federate fuses the legs by rank because they share no vector space: the vault em
   86  Late chunking › Method
   18  Late chunking › Results
  234  repo › /Users/71293/code/personal/orgs/vishalakshi/vishalakshi/code.py:57
  154  grep › index.ipynb:161
  600  repo › /Users/71293/code/personal/orgs/vishalakshi/vishalakshi/extract.py:179
   89  grep › 07_concepts.ipynb:101


In [ ]:
for s in c.related: print(f'{s.via:6}  {s.breadcrumb}')

`results` answer the question. `related` are associations along the entity graph, useful before you know what to search for, not wired into ranking.


In [ ]:
c.encoder

'minishlab/potion-multilingual-128M (256d, float16, model2vec)'

`c.encoder` always says which embedder answered. Hashing fallback costs +0.089 recall@10 on known-item queries (`evals/encoder.py`).


## Models and backends

Naming a model is naming it to rishi. LiteRT is the default local path; hosted ids need their keys. Details that change with rishi stay there; this page only shows what the vault passes through.


In [ ]:
import rishi.core
from urai import Chat, infer_runtime
from vishalakshi.ask import dflt_model

# Rishi names and registers the backends. Urai resolves their model ids.
gemma4_e2b = 'litert-community/gemma-4-E2B-it-litert-lm'
dflt_model, gemma4_e2b     # $VISHALAKSHI_MODEL if set, else the small local default

In [ ]:
{m: infer_runtime(m) for m in (gemma4_e2b, 'mlx-community/Qwen3-4B-4bit', 'my-local.gguf',
                               'gpt-5.6-luna', 'gemma-3-4b-it-int4')}

In [ ]:
#| eval: false
v.ask('why are rankings fused?', model=gemma4_e2b)             # an id: rishi knows what runs it
v.ask('why are rankings fused?', model='llama/my-local.gguf')  # a prefix when the id cannot say
v.ask('why are rankings fused?', model='mlx-community/Qwen3-4B-4bit',
      chat_kw=dict(temp=0, think=True))          # the rest of rishi's constructor

In [ ]:
from fastcore.test import test_fail
# `model=` plus `chat_kw=` is the model API. Urai reports an id that names no registered backend.
test_fail(lambda: Chat('gemma-3-4b-it-int4'), contains='backend')

A reasoning model's `<think>` block goes to `r.thinking` rather than `r.answer`: it names sections
it then discards, and citations are read off the answer.

Retrieval never needs the network. Only answering with a hosted model does.

## Encoders

Default is a static multilingual encoder, 256d float16. `offline=True` hashes instead of embedding (+0.089 recall@10 on known-item queries, `evals/encoder.py`). A shelf records the encoder that wrote it.


In [ ]:
Vault(':memory:', offline=True).enc.note      # never attempt a download; also the CI default

'char-n-gram hashing (256d): lexical only; pass encoder= or restore network access for real semantics'

In [ ]:
#| eval: false
Vault(encoder='minishlab/potion-science-32M')   # pick a different one

Vault('/Users/71293/.vishalakshi/vault.db': 0 docs, 0 chunks, 0 entities, encoder=model2vec)

Float16 is litesearch's default width. One ANN index is one vector space: different encoders need different shelves.


Three aliases, each a plain model id. Any other model2vec id works by name too.

In [ ]:
from vishalakshi.core import ENCODERS

ENCODERS

Static models are lookup tables: milliseconds per document, no GPU. Across four encoders litesearch measures 0.018 to 0.046 weighted MRR, so the static default is the measured trade-off (`evals/encoder.py`). `gemma` is the ONNX one, and asking for it is what pulls onnxruntime in.

In [ ]:
# `encoder=` also takes any object with an `.encode`, which is what a shelf gets when you build one
import numpy as np
from vishalakshi.core import mk_encoder

class Eight:
    def encode(self, xs, **kw): return np.zeros((len(xs), 8), dtype=np.float16)

e = mk_encoder(Eight())
e.dims, e.method, e.name

One ANN index is one vector space. Different encoders need different shelves; `SHELVES` names the common ones, `KIND_SHELF` routes acquisition.


In [ ]:
from vishalakshi.core import SHELVES, KIND_SHELF

SHELVES, KIND_SHELF

In [ ]:
papers = v.shelf('papers')        # reuses the live encoder unless a registry entry exists
papers.add('# Late chunking\n\nWe evaluate contextual chunk embeddings on BEIR.', 'a paper')

[(s['store'], s['encoder'], s['docs']) for s in v.shelves()]


`KIND_SHELF` routes acquisition. `reshelf` moves a document after the fact; `elsewhere` searches neighbouring shelves.


In [ ]:
v.route('arxiv').name, v.route('web').name

('papers', 'store')

A PDF may be an invoice or a paper. `categorize` decides after filing; `reshelf` moves it.


In [ ]:
v.add('# Attention is all you need\n\n## Abstract\n\nWe propose the Transformer.\n\n'
      '## Introduction\n\nRelated work [1] et al.\n\n## References\n\ndoi:10.1/x',
      'attention', source='/inbox/attention.pdf')

r = v.reshelf('/inbox/attention.pdf', llm='never')
r.doctype, r.was, r.store, r.moved

('paper', 'store', 'papers', True)

In [ ]:
v.doc('/inbox/attention.pdf'), v.shelf('papers').doc('/inbox/attention.pdf')['title']

(None, 'attention')

A move re-ingests: vectors are remade for the destination encoder. Durable meta lives in `doc_marks`.


`federate` fuses shelves by rank with kosha and ripgrep. `elsewhere` appends a couple of neighbour sections for `context` / `ask`.


In [ ]:
v.federate('contextual chunk embeddings', repo=False, grep=False).legs

{'prose': 3, 'shelf:papers': 4}

`elsewhere` tags the source shelf. Read with `store=` when the `node_id` is not on the current shelf.


In [ ]:
e = v.elsewhere('contextual chunk embeddings')
L(e).map(lambda r: (r.store, r.breadcrumb))


[('papers', 'papers › a paper'),
 ('papers', 'papers › attention › Attention is all you need › Abstract')]

In [ ]:
v.read(e[0].node_id, store=e[0].store)['text'][:80]

'# Late chunking\n\nWe evaluate contextual chunk embeddings on BEIR.'

A shelf records the encoder that wrote it, reopens with the right one, and warns on mismatch. Rebuild the shelf if you change encoders on purpose.
